# Ticha HTR — Zapotec language model fusion (no training)

This notebook tests whether the Zapotec text-only decoders from `ticha_trocr_finetuning-modified-transformer.ipynb` (Part 1) help the best end-to-end model, **without training anything**. The end-to-end model is `trocr-base-handwritten-finetuned-smooth-cosine-aug` (fine-tuned on augmented lines, validation CER 0.165 during training).

The language model (LM) is added at decoding time, in two ways:

1. **N-best rescoring.** The HTR model proposes `NBEST` candidate lines with beam search. Each candidate gets `(log p_HTR(line | image) + β · log p_LM(line)) / tokens`, an average per token as in beam search itself (a plain sum would favor short, cut-off lines). The candidates are generated once, so every `β` can be tried in seconds.
2. **Shallow fusion.** The LM is added inside the beam search at every step: `log p_HTR(token | image, prefix) + β · log p_LM(token | prefix)`. It can find lines that plain beam search never proposes, but every `β` needs a new decoding pass, so `β` is tuned on a subset of validation lines.

`β = 0` is the HTR model alone (beam search's own first choice). `β` is chosen on validation; the test split is decoded once at the end.

Both LMs (`zap`, `zap_spa`) use the same RoBERTa tokenizer as the HTR model, so their tokens line up. They were trained with a vocabulary mask (tokens never seen in the text corpus get −10⁴); here a log-probability floor `LM_FLOOR` stops one unseen word piece from vetoing a line.

**Runs locally** (Apple Silicon GPU through MPS, or CPU), with the `Python (ticha)` kernel. The models are copied from the VM into `data/models/` (git-ignored, about 3.2 GB) by the cell below if they are missing.

In [1]:
import os
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')  # run ops MPS lacks on the CPU instead of failing
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
import io
import json
import re
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from jiwer import cer, wer
from PIL import Image
from transformers import LogitsProcessor, LogitsProcessorList, TrOCRProcessor, VisionEncoderDecoderModel
from transformers.models.trocr.modeling_trocr import TrOCRForCausalLM

# The kernel may start in the repo root or in notebooks/.
ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
MODELS = ROOT / 'data' / 'models'
HTR_DIR = MODELS / 'e2e-aug'
LM_PATHS = {variant: MODELS / f'decoder_{variant}' / 'decoder_state.pt' for variant in ['zap', 'zap_spa']}
DATA_DIR = ROOT / 'data' / 'hf_dataset_v1.1' / 'original'  # validation and test are the same as on the Hub
OUTPUT_DIR = ROOT / 'data' / 'evaluation' / 'lm_fusion'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(os.environ.get('TICHA_DEVICE')
                      or ('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'))
BATCH_SIZE = 4           # images per generate call
NBEST = 5                # beam width and candidates per line for rescoring
MAX_LENGTH = 64          # the model's generation max_length
LM_FLOOR = -12.0         # lowest LM log-probability per token (about log 1/50k); see the intro
RESCORE_BETAS = [0.0, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0]
FUSION_BETAS = [0.05, 0.1, 0.2, 0.3]
FUSION_VAL_LINES = 200   # validation lines used to choose beta for shallow fusion
START_CHECK_LINES = 150  # validation lines used to check the decoder start token
COMPARABLE_EXCLUDE = {'Za687'}  # same pages as Th687, which older train splits contain
print('Device:', DEVICE, '| repo:', ROOT)

/opt/homebrew/anaconda3/envs/ticha/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps | repo: /Users/paw/macss/ticha


## Copy the models from the VM (only if missing)

The end-to-end model and the two text-only decoders live on `ticha-l4`. This cell copies them with `gcloud compute scp` when they are not in `data/models/` yet. Check the free disk space first: about 3.2 GB are needed.

In [2]:
VM = ['--project', 'dml-course-511723', '--zone', 'us-central1-a']
HTR_FILES = ['config.json', 'generation_config.json', 'model.safetensors', 'preprocessor_config.json',
             'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json', 'merges.txt']
remote_htr = '/home/paw/ticha-runs/trocr-base-handwritten-finetuned-smooth-cosine-aug'
copies = [(f'{remote_htr}/{name}', HTR_DIR / name) for name in HTR_FILES]
copies += [(f'/home/paw/ticha-runs/trocr-modified/decoder_{variant}/decoder_state.pt', path) for variant, path in LM_PATHS.items()]
for remote, local in copies:
    if local.exists():
        continue
    local.parent.mkdir(parents=True, exist_ok=True)
    print('Copying', remote)
    subprocess.run(['gcloud', 'compute', 'scp', *VM, f'ticha-l4:{remote}', str(local)], check=True)
print('Models present:', all(local.exists() for _, local in copies))

Models present: True


## Validation and test lines

The same cleanup as the other notebooks (`clean_text`, copied from `ticha_trocr_finetuning-modified-transformer.ipynb`): abbreviation expansions and glosses removed, lines with damage notes or uncertain readings dropped, so the scores are comparable.

In [3]:
BRACKET_RE = re.compile(r'\[([^\[\]]*)\]|\(([^()]*)\)')
NOTE_RE = re.compile(
    r'[?…]|^\.+$|\.\.|\s|,|^x+$|^word$'
    r'|cortado|roto|torn|mancha|borrado|firmas|ilegible',
    re.IGNORECASE)
DROP_HASH = True
GUESS_RE = re.compile(r'\((?!word\?)([^()?]*[^\W\d_][^()?]*)\?+\)')


def clean_text(text, keep_guesses=False):
    """Return the diplomatic text, or None when the line should be dropped."""
    text = str(text or '')
    if keep_guesses:
        text = GUESS_RE.sub(lambda m: m.group(1), text)
    notes = [square or round for square, round in BRACKET_RE.findall(text)]
    if any(NOTE_RE.search(note.strip()) for note in notes if not note.strip().startswith('=')):
        return None
    text = BRACKET_RE.sub('', text)
    if re.search(r'[\[\]()]', text) or (DROP_HASH and '#' in text):
        return None
    return re.sub(r'\s+', ' ', text).strip() or None


def load_split(name):
    df = pd.concat([pd.read_parquet(p) for p in sorted(DATA_DIR.glob(f'{name}-*.parquet'))], ignore_index=True)
    df['text'] = df['text'].map(clean_text)
    kept = df[df['text'].notna()]
    if name == 'test':  # Za687 is the same manuscript as Th687, which the v1.0 train split contains
        kept = kept[~kept['doc_id'].isin(COMPARABLE_EXCLUDE)]
    kept = kept.reset_index(drop=True)
    print(f'{name}: kept {len(kept):,} / {len(df):,} lines')
    return kept


splits = {name: load_split(name) for name in ['validation', 'test']}


def line_image(row):
    return Image.open(io.BytesIO(row['image']['bytes'])).convert('RGB')

validation: kept 710 / 819 lines
test: kept 973 / 1,066 lines


## Load the HTR model and check the decoder start token

The model was trained with decoder inputs that start with token `0` (`config.json`), but its saved `generation_config.json` says `decoder_start_token_id: 2`, and that is the one `generate()` uses. This cell decodes `START_CHECK_LINES` validation lines with each start token and keeps the better one for the rest of the notebook. If `0` wins, the CERs reported for the TrOCR models so far were measured with the wrong start token.

In [4]:
processor = TrOCRProcessor.from_pretrained(HTR_DIR)
tokenizer = processor.tokenizer
htr = VisionEncoderDecoderModel.from_pretrained(HTR_DIR).to(DEVICE).eval()
PAD, EOS, BOS = tokenizer.pad_token_id, tokenizer.sep_token_id, tokenizer.cls_token_id
print('config start token:', htr.config.decoder_start_token_id,
      '| generation_config start token:', htr.generation_config.decoder_start_token_id)


def batches(df, size=BATCH_SIZE):
    for start in range(0, len(df), size):
        chunk = df.iloc[start:start + size]
        pixel_values = processor(images=[line_image(r) for _, r in chunk.iterrows()], return_tensors='pt').pixel_values
        yield chunk, pixel_values.to(DEVICE)


def decode(sequences):
    return [t.strip() for t in tokenizer.batch_decode(sequences, skip_special_tokens=True)]


@torch.inference_mode()
def generate_lines(df, num_beams, start_token, logits_processor=None):
    lines = []
    for _, pixel_values in batches(df):
        sequences = htr.generate(pixel_values=pixel_values, num_beams=num_beams, max_length=MAX_LENGTH,
                                 early_stopping=True, decoder_start_token_id=start_token,
                                 logits_processor=logits_processor)
        lines += decode(sequences)
    return lines


check = splits['validation'].iloc[:START_CHECK_LINES]
generate_lines(check.iloc[:BATCH_SIZE], 2, 0)  # warm-up, so the first timing doesn't include it
start_cer = {}
for start_token in sorted({htr.config.decoder_start_token_id, htr.generation_config.decoder_start_token_id}):
    t0 = time.time()
    start_cer[start_token] = cer(list(check['text']), generate_lines(check, 2, start_token))
    seconds_per_line = (time.time() - t0) / len(check)
    print(f'start token {start_token}: CER {start_cer[start_token]:.4f}  ({seconds_per_line:.2f} s/line, beam 2)')
START_TOKEN = min(start_cer, key=start_cer.get)
print('Using start token', START_TOKEN)

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "transformers_version": "4.46.3"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder

config start token: 0 | generation_config start token: 2
start token 0: CER 0.1892  (1.19 s/line, beam 2)
start token 2: CER 0.1889  (1.33 s/line, beam 2)
Using start token 2


## The Zapotec language models

Each LM is the TrOCR decoder trained on text only, loaded as a `TrOCRForCausalLM` without cross-attention inputs (as in Part 1). Its input starts with token `0`, as in training, whatever start token the HTR model uses.

In [5]:
def load_lm(variant):
    state = torch.load(LM_PATHS[variant], map_location='cpu', weights_only=True)
    mask = state.pop('output_projection.vocab_mask')
    lm = TrOCRForCausalLM(htr.config.decoder)
    lm.load_state_dict(state)
    lm.vocab_mask = mask.to(DEVICE)
    return lm.to(DEVICE).eval()


def lm_log_probs(lm, input_ids):
    """Floored LM log-probabilities of the next token after each prefix; the first token is set to 0, as in training."""
    input_ids = input_ids.clone()
    input_ids[:, 0] = BOS
    logits = lm(input_ids=input_ids).logits.float() + lm.vocab_mask
    return logits.log_softmax(-1).clamp(min=LM_FLOOR)

## 1. N-best rescoring

Beam search with `NBEST` beams returns `NBEST` candidates per line. For each candidate the notebook stores the HTR log-probability (teacher-forced, given the image), each LM's log-probability, and the number of tokens. The candidates are cached in `data/evaluation/lm_fusion/`, so rerunning the sweep doesn't decode again.

In [6]:
def target_mask(sequences):
    """Positions that are real target tokens: everything after the start token up to and including the first </s>."""
    targets = sequences[:, 1:]
    ended_before = torch.cumsum((targets == EOS).int(), dim=1) - (targets == EOS).int()
    return (ended_before == 0) & (targets != PAD)


@torch.inference_mode()
def nbest(split_name):
    path = OUTPUT_DIR / f'nbest_{split_name}_start{START_TOKEN}.json'
    if path.exists():
        return json.loads(path.read_text())
    df, rows, t0 = splits[split_name], [], time.time()
    for done, (chunk, pixel_values) in enumerate(batches(df), 1):
        sequences = htr.generate(pixel_values=pixel_values, num_beams=NBEST, num_return_sequences=NBEST,
                                 max_length=MAX_LENGTH, early_stopping=True, decoder_start_token_id=START_TOKEN)
        encoder_states = htr.encoder(pixel_values=pixel_values).last_hidden_state.repeat_interleave(NBEST, 0)
        logits = htr(encoder_outputs=(encoder_states,), decoder_input_ids=sequences[:, :-1]).logits.float()
        mask = target_mask(sequences)
        token_logp = logits.log_softmax(-1).gather(-1, sequences[:, 1:].unsqueeze(-1)).squeeze(-1)
        htr_logp = (token_logp * mask).sum(-1).cpu().tolist()
        lengths = mask.sum(-1).cpu().tolist()
        texts = decode(sequences)
        for i, (_, row) in enumerate(chunk.iterrows()):
            rows.append({'id': row['id'], 'candidates': [
                {'text': texts[i * NBEST + k], 'ids': sequences[i * NBEST + k].cpu().tolist(),
                 'htr': htr_logp[i * NBEST + k], 'tokens': lengths[i * NBEST + k]} for k in range(NBEST)]})
        if done % 25 == 0:
            elapsed = time.time() - t0
            print(f'{split_name}: {len(rows)}/{len(df)} lines, {elapsed / len(rows):.2f} s/line', flush=True)
    path.write_text(json.dumps(rows))
    return rows


@torch.inference_mode()
def add_lm_scores(rows, variant, lm):
    """Adds each candidate's summed LM log-probability under `variant`."""
    if all(variant in c for row in rows for c in row['candidates']):
        return
    candidates = [c for row in rows for c in row['candidates']]
    for start in range(0, len(candidates), 32):
        chunk = candidates[start:start + 32]
        width = max(len(c['ids']) for c in chunk)
        sequences = torch.full((len(chunk), width), PAD, dtype=torch.long)
        for i, c in enumerate(chunk):
            sequences[i, :len(c['ids'])] = torch.tensor(c['ids'])
        sequences = sequences.to(DEVICE)
        token_logp = lm_log_probs(lm, sequences[:, :-1]).gather(-1, sequences[:, 1:].unsqueeze(-1)).squeeze(-1)
        for c, value in zip(chunk, (token_logp * target_mask(sequences)).sum(-1).cpu().tolist()):
            c[variant] = value


nbest_rows = {name: nbest(name) for name in ['validation', 'test']}
for variant in LM_PATHS:
    lm = load_lm(variant)
    for name, rows in nbest_rows.items():
        add_lm_scores(rows, variant, lm)
        (OUTPUT_DIR / f'nbest_{name}_start{START_TOKEN}.json').write_text(json.dumps(rows))
    del lm
print('Candidates scored with', list(LM_PATHS))

validation: 100/710 lines, 3.63 s/line
validation: 200/710 lines, 3.31 s/line
validation: 300/710 lines, 3.24 s/line
validation: 400/710 lines, 3.23 s/line
validation: 500/710 lines, 3.27 s/line
validation: 600/710 lines, 3.28 s/line
validation: 700/710 lines, 3.34 s/line
test: 100/973 lines, 3.33 s/line
test: 200/973 lines, 3.24 s/line
test: 300/973 lines, 3.22 s/line
test: 400/973 lines, 3.43 s/line
test: 500/973 lines, 3.47 s/line
test: 600/973 lines, 3.46 s/line
test: 700/973 lines, 3.45 s/line
test: 800/973 lines, 3.42 s/line
test: 900/973 lines, 3.35 s/line
Candidates scored with ['zap', 'zap_spa']


In [7]:
def pick(rows, variant=None, beta=0.0):
    """Best candidate per line under (log p_HTR + beta * log p_LM) / tokens; without an LM, beam search's first choice."""
    if not variant or not beta:
        return [row['candidates'][0]['text'] for row in rows]

    def score(c):
        return (c['htr'] + beta * c[variant]) / max(c['tokens'], 1)
    return [max(row['candidates'], key=score)['text'] for row in rows]


val_refs = list(splits['validation']['text'])
sweep = {}
for variant in LM_PATHS:
    for beta in RESCORE_BETAS:
        sweep[(variant, beta)] = cer(val_refs, pick(nbest_rows['validation'], variant, beta))
sweep_frame = pd.Series(sweep).rename_axis(['lm', 'beta']).unstack('lm')
print('Validation CER of N-best rescoring (rows: beta; columns: LM). beta = 0 is the HTR model alone.')
display(sweep_frame.round(4))

baseline_val = sweep[(next(iter(LM_PATHS)), 0.0)]
best_rescore = min(sweep, key=sweep.get)
print(f'HTR alone (beam {NBEST}): {baseline_val:.4f}')
print(f'Best rescoring: lm={best_rescore[0]}, beta={best_rescore[1]} -> {sweep[best_rescore]:.4f}')

Validation CER of N-best rescoring (rows: beta; columns: LM). beta = 0 is the HTR model alone.


lm,zap,zap_spa
beta,,
0.00,0.1611,0.1611
0.10,0.1627,0.1627
0.20,0.1631,0.1626
0.30,0.1641,0.1632
0.50,0.1669,0.1652
0.75,0.1677,0.1657
1.00,0.1683,0.1662


HTR alone (beam 5): 0.1611
Best rescoring: lm=zap, beta=0.0 -> 0.1611


## 2. Shallow fusion

A `LogitsProcessor` adds `β · log p_LM(next token | prefix)` to the HTR model's log-probabilities at each beam-search step (Transformers passes log-probabilities to the processors in beam search). The LM recomputes the whole prefix each step; the lines are short, so that is fast enough. Beam search ranks finished lines by average log-probability (`length_penalty = 1`), as the rescoring does.

The LM is the one that did best in rescoring (with `β > 0`). `β` is chosen on the first `FUSION_VAL_LINES` validation lines. Those lines are harder than the validation average (the HTR model alone scores about 0.23 on them, against 0.16 on all of validation), so the cell prints the `β = 0` CER on the same lines for comparison.

In [8]:
class ShallowFusion(LogitsProcessor):
    def __init__(self, lm, beta):
        self.lm, self.beta = lm, beta

    @torch.inference_mode()
    def __call__(self, input_ids, scores):
        return scores + self.beta * lm_log_probs(self.lm, input_ids)[:, -1, :].to(scores.dtype)


fusion_variant = min(LM_PATHS, key=lambda v: min(sweep[(v, b)] for b in RESCORE_BETAS if b))
fusion_lm = load_lm(fusion_variant)
fusion_val = splits['validation'].iloc[:FUSION_VAL_LINES]
fusion_refs = list(fusion_val['text'])
fusion_cer = {0.0: cer(fusion_refs, pick(nbest_rows['validation'][:FUSION_VAL_LINES]))}  # beam NBEST, no LM
# These lines are harder than the validation average, so compare against this, not the full-validation CER.
print(f'beta=0 (HTR alone, beam {NBEST}) on these {len(fusion_val)} lines: CER {fusion_cer[0.0]:.4f}', flush=True)
for beta in FUSION_BETAS:
    t0 = time.time()
    lines = generate_lines(fusion_val, NBEST, START_TOKEN, LogitsProcessorList([ShallowFusion(fusion_lm, beta)]))
    fusion_cer[beta] = cer(fusion_refs, lines)
    print(f'beta={beta}: CER {fusion_cer[beta]:.4f} ({(time.time() - t0) / len(fusion_val):.2f} s/line)', flush=True)
best_fusion_beta = min(fusion_cer, key=fusion_cer.get)
print(f'Shallow fusion with {fusion_variant}: best beta on {len(fusion_val)} validation lines = {best_fusion_beta}')

beta=0.1: CER 0.2339 (4.03 s/line)
beta=0.2: CER 0.2393 (4.00 s/line)
beta=0.3: CER 0.2487 (4.02 s/line)
beta=0.5: CER 0.2696 (4.01 s/line)
Shallow fusion with zap_spa: best beta on 200 validation lines = 0.0


## Test evaluation

Decoded once, with the settings chosen on validation:

- **HTR alone**: beam search with `NBEST` beams, best candidate by HTR score.
- **N-best rescoring** with the best LM and `β`.
- **Shallow fusion** with the best `β` (skipped if `β = 0` was best on validation).

Scores are broken down by language, and repeated without Za687 for comparison with `ticha_trocr_evaluation.ipynb`.

In [9]:
test_df = splits['test']
test_predictions = {
    'HTR alone': pick(nbest_rows['test']),
    f'rescoring {best_rescore[0]} (beta={best_rescore[1]})': pick(nbest_rows['test'], *best_rescore),
}
if best_fusion_beta:
    test_predictions[f'fusion {fusion_variant} (beta={best_fusion_beta})'] = generate_lines(
        test_df, NBEST, START_TOKEN, LogitsProcessorList([ShallowFusion(fusion_lm, best_fusion_beta)]))


def scores_by_language(name, df):
    rows = []
    for language, group in [('all', df), *df.groupby('language')]:
        rows.append({'model': name, 'language': language, 'lines': len(group),
                     'cer': cer(list(group['reference']), list(group['prediction'])),
                     'wer': wer(list(group['reference']), list(group['prediction']))})
    return rows


frames = {}
for name, predictions in test_predictions.items():
    frames[name] = pd.DataFrame({'id': test_df['id'], 'doc_id': test_df['doc_id'], 'language': test_df['language'],
                                 'reference': test_df['text'], 'prediction': predictions})
    frames[name].to_csv(OUTPUT_DIR / f"test_{re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_')}.csv", index=False)

for title, keep in [('Without Za687 (comparable with ticha_trocr_evaluation.ipynb)',
                     lambda df: df[~df['doc_id'].isin(COMPARABLE_EXCLUDE)])]:
    scores = pd.DataFrame([row for name, df in frames.items() for row in scores_by_language(name, keep(df))])
    print(title)
    display(scores.pivot(index='model', columns='language', values='cer').round(4))
scores.to_csv(OUTPUT_DIR / 'test_scores_without_Za687.csv', index=False)

[Removed on 2026-10-17: test metrics measured on all 973 test lines, including Za687, which duplicates the train document Th687. The tables without Za687 are kept; see the report, section 9.]


Without Za687 (comparable with ticha_trocr_evaluation.ipynb)


language,Spanish,Spanish with some Zapotec,Zapotec,all
model,,,,
HTR alone,0.1068,0.2112,0.1671,0.1437
rescoring zap (beta=0.0),0.1068,0.2112,0.1671,0.1437


In [10]:
# Lines the language model changed, for a look at what it fixes and what it breaks.
changed_name = list(frames)[-1]
diff = frames['HTR alone'][['reference']].assign(htr=frames['HTR alone']['prediction'],
                                                 with_lm=frames[changed_name]['prediction'])
diff = diff[diff['htr'] != diff['with_lm']]
print(f'{changed_name}: {len(diff)} of {len(diff.index.union(frames["HTR alone"].index))} test lines changed')
pd.set_option('display.max_colwidth', 80)
display(diff.head(25))

rescoring zap (beta=0.0): 0 of 973 test lines changed


,reference,htr,with_lm


## How to read the results

- `β = 0` rows are the HTR model alone; any gain has to show up against them, on validation first.
- If rescoring helps but fusion doesn't (or the reverse), report both: rescoring can only choose among the HTR model's own candidates, fusion can produce new lines.
- Compare the **Zapotec** column: the LMs were trained on Zapotec text, so that is where they should help; the Spanish column shows what they cost.
- State the start-token check: if start token `0` beat `2`, the earlier TrOCR CERs were measured with the wrong start token, and the fair baseline is the one here.
- `LM_FLOOR` and the vocabulary mask: word pieces never seen in the text corpus get the floor instead of −10⁴.